# Module 15: Publication-Quality Figures

## Journal-Ready, Accessible, and High-Resolution Visualization

This module covers everything you need to create **publication-quality figures** suitable for scientific journals, reports, and presentations. Using CHIRPS rainfall data, you will learn about figure sizing for journals, high-resolution export, vector versus raster formats, colorblind-friendly design, and a complete checklist for submission-ready graphics.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import warnings; warnings.filterwarnings('ignore')

ds = xr.open_dataset('../chirps_1981_2022.nc', engine='netcdf4')
precip = ds['precip']

eth_ts = precip.sel(lat=slice(15.0, 3.0), lon=slice(33.0, 48.0)).mean(dim=['lat', 'lon'])
rainfall_series = eth_ts.to_series()

print(f'Data loaded: {len(rainfall_series)} months')
print(f'Matplotlib version: {mpl.__version__}')

## 2. Figure Sizing for Journals

Scientific journals specify exact figure dimensions. Common standards:

| Journal | Column width | Page width |
|---------|-------------|------------|
| Nature  | 89 mm (3.5 in) | 183 mm (7.2 in) |
| Science | 87 mm (3.4 in) | 178 mm (7.0 in) |
| AGU     | 84 mm (3.3 in) | 174 mm (6.85 in) |
| EGU     | 86 mm (3.4 in) | 177 mm (6.97 in) |
| PLOS ONE | — | 174 mm (6.85 in) |

Use the **golden ratio** (~1.618) for aspect ratio, or 4:3 / 16:9 for presentations.

In [ ]:
# Single-column figure (Nature: 89 mm = 3.5 in)
# Golden ratio height: 3.5 / 1.618 = 2.16 in
single_col_w = 3.5  # inches
single_col_h = single_col_w / 1.618

fig, ax = plt.subplots(figsize=(single_col_w, single_col_h), dpi=150)
ax.plot(rainfall_series.index, rainfall_series.values, color='black', lw=0.5)
ax.set_title('Single-Column Figure (89 mm)', fontsize=8)
ax.set_xlabel('Year', fontsize=7)
ax.set_ylabel('Precipitation (mm/month)', fontsize=7)
ax.tick_params(labelsize=6)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Size: {single_col_w:.1f} x {single_col_h:.1f} in = {single_col_w*25.4:.0f} x {single_col_h*25.4:.0f} mm')

In [ ]:
# Double-column / full-page width (Nature: 183 mm = 7.2 in)
double_col_w = 7.2
double_col_h = double_col_w / 1.618

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(double_col_w, double_col_h * 0.6), dpi=150)

annual = rainfall_series.resample('YE').sum()

ax1.plot(annual.index, annual.values, 'o-', color='steelblue', lw=0.8, markersize=2)
ax1.set_title('Annual Total', fontsize=9)
ax1.set_xlabel('Year', fontsize=7)
ax1.set_ylabel('mm/year', fontsize=7)
ax1.tick_params(labelsize=6)
ax1.grid(True, alpha=0.3)

monthly_clim = rainfall_series.groupby(rainfall_series.index.month).mean()
ax2.bar(monthly_clim.index, monthly_clim.values, color='steelblue', width=0.7)
ax2.set_title('Monthly Climatology', fontsize=9)
ax2.set_xlabel('Month', fontsize=7)
ax2.set_ylabel('mm/month', fontsize=7)
ax2.tick_params(labelsize=6)
ax2.set_xticks(range(1, 13))
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Size: {double_col_w:.1f} x {double_col_h*0.6:.1f} in')

## 3. High-Resolution Export (DPI, Formats)

Journals typically require:
- **Minimum 300 DPI** (600+ for fine detail)
- **CMYK color** for print journals
- **Vector formats** (SVG, EPS, PDF) for text and line art
- **TIFF or high-quality PNG** for raster images

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.5), dpi=150)
ax.plot(rainfall_series.index, rainfall_series.values, color='steelblue', lw=0.4)
ax.set_title('High-Resolution Export Demo', fontsize=8)
ax.set_ylabel('mm/month', fontsize=7)
ax.tick_params(labelsize=6)
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Export to multiple formats
fig.savefig('exports/rainfall_series.png', dpi=300, bbox_inches='tight')
fig.savefig('exports/rainfall_series.pdf', dpi=300, bbox_inches='tight')
fig.savefig('exports/rainfall_series.svg', dpi=300, bbox_inches='tight')
fig.savefig('exports/rainfall_series.eps', dpi=300, bbox_inches='tight')
print('Exported to exports/ directory')
plt.show()

## 4. SVG, PNG, PDF, EPS — When to Use Which

| Format | Type | Best For | Notes |
|--------|------|----------|-------|
| **PNG** | Raster | Web, screenshots, presentations | Lossless, large file size at high DPI |
| **SVG** | Vector | Web, editing in Illustrator/Inkscape | Editable, infinite zoom, large with complex data |
| **PDF** | Vector | Journal submission, printing | Embeddable fonts, reliable rendering |
| **EPS** | Vector | LaTeX (via `\includegraphics`), older journals | Legacy format, less compact than PDF |
| **TIFF** | Raster | Some journals (e.g., Nature) | Supports LZW compression, CMYK |

In [ ]:
# Demonstrate rasterization for vector exports (large datasets)
fig, ax = plt.subplots(figsize=(7, 2.5), dpi=150)

# Use rasterized=True for the line to reduce vector file size
ax.plot(rainfall_series.index, rainfall_series.values, color='black', lw=0.4, rasterized=True)
ax.set_title('Rasterized line in vector export (smaller PDF)', fontsize=9)
ax.set_ylabel('mm/month', fontsize=8)
ax.tick_params(labelsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig('exports/rainfall_rasterized.pdf', dpi=300, bbox_inches='tight')
print('Saved: exports/rainfall_rasterized.pdf')
plt.show()

## 5. Accessibility in Visualization

Accessible design ensures your figures are readable by everyone, including those with color vision deficiency (CVD).

### Key Principles:
1. **Don't rely solely on color** — use patterns, markers, and labels
2. **Sufficient contrast** — text against background
3. **Font sizes** — at least 7 pt for print, 12 pt for presentations
4. **Universal design** — avoid flashing, use clear hierarchies

In [ ]:
# Simulate common color vision deficiencies
from matplotlib.colors import hsv_to_rgb, rgb_to_hsv

def simulate_cvd(image, deficiency='deuteranopia'):
    """Simulate color vision deficiency on an RGB image."""
    # Simplified simulation using color blindness matrices
    # Full simulation would use daltonization; this is a basic approximation
    if deficiency == 'deuteranopia':
        # Red-green blindness (green cone missing)
        sim = image.copy()
        sim[:, :, 1] = image[:, :, 0] * 0.625 + image[:, :, 1] * 0.375
        return sim
    elif deficiency == 'protanopia':
        # Red-green blindness (red cone missing)
        sim = image.copy()
        sim[:, :, 0] = image[:, :, 1] * 0.8 + image[:, :, 0] * 0.2
        return sim
    return image

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

grad = np.linspace(0, 1, 256).reshape(1, -1)
cmaps = ['rainbow', 'viridis', 'turbo']
titles = ['Rainbow', 'Viridis (colorblind-friendly)', 'Turbo']

for i, (cm, title) in enumerate(zip(cmaps, titles)):
    axes[i].imshow(grad, aspect='auto', cmap=cm)
    axes[i].set_title(title, fontsize=9)
    axes[i].set_xticks([])
    axes[i].set_yticks([])

plt.tight_layout()
plt.show()
print('Rainbow and Turbo are NOT colorblind-friendly; Viridis is.')

## 6. Colorblind-Friendly Palettes

Use perceptually uniform sequential colormaps: **viridis**, **plasma**, **inferno**, **magma**, **cividis**.

For categorical data, use the **colorblind-safe palettes** from sources like Wong (2011) or Tol (2021).

In [ ]:
# Test colormaps on rainfall map
jan_data = precip.sel(time='2020-01-01').values

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

test_cmaps = ['viridis', 'plasma', 'inferno', 'magma', 'cividis', 'Blues']

for ax, cmap in zip(axes.flat, test_cmaps):
    im = ax.imshow(jan_data, cmap=cmap, aspect='auto',
                   extent=[30, 60, 0, 20], origin='upper')
    ax.set_title(cmap, fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8, ticks=[])

plt.suptitle('Colorblind-Friendly Colormap Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Create a custom colorblind palette for categorical data
# Wong (2011) Nature Methods palette
wong_palette = [
    '#000000',  # Black
    '#E69F00',  # Orange
    '#56B4E9',  # Sky Blue
    '#009E73',  # Bluish Green
    '#F0E442',  # Yellow
    '#0072B2',  # Blue
    '#D55E00',  # Vermillion
    '#CC79A7',  # Reddish Purple
]

# Plot using the Wong palette
regions = ['North', 'South', 'East', 'West', 'Central']
fig, ax = plt.subplots(figsize=(7, 4))

annual = rainfall_series.resample('YE').sum()
x = annual.index.year
colors = [wong_palette[i % len(wong_palette)] for i in range(len(regions))]

# Demonstrate with regional data
for i, region in enumerate(regions):
    lat_slice = slice(15.0 - i*2, 13.0 - i*2)
    lon_slice = slice(33.0 + i*2, 35.0 + i*2)
    try:
        region_ts = precip.sel(lat=lat_slice, lon=lon_slice).mean(dim=['lat','lon']).to_series()
        region_annual = region_ts.resample('YE').sum()
        ax.plot(region_annual.index.year, region_annual.values,
                color=wong_palette[i], lw=1.2, label=region, marker='o', markersize=3)
    except:
        pass

ax.set_title('Wong (2011) Colorblind-Safe Palette', fontsize=12)
ax.set_xlabel('Year')
ax.set_ylabel('Annual Rainfall (mm/year)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Journal-Ready Graphics Checklist

Before submission, verify each item:

### Layout & Dimensions
- [ ] Figure dimensions match journal column width
- [ ] Consistent aspect ratio across multi-panel figures
- [ ] Sufficient white space between panels
- [ ] All panels labeled (A, B, C, ...) in matching font

### Typography
- [ ] Font size ≥ 7 pt for print, ≥ 12 pt for presentations
- [ ] Consistent font family (prefer sans-serif for readability)
- [ ] Axis labels legible and descriptive
- [ ] No overlapping text or labels

### Color & Accessibility
- [ ] Colorblind-friendly palette used (not rainbow or jet)
- [ ] Sufficient contrast between data and background
- [ ] Information conveyed without color alone
- [ ] Colorbar present for all mapped quantities

### Export
- [ ] Resolution ≥ 300 DPI
- [ ] CMYK color for print journals
- [ ] Vector format (PDF/EPS) for line art
- [ ] Fonts embedded or converted to outlines
- [ ] Bounding box tight (no excess whitespace)

In [ ]:
# Apply the full checklist to create a journal-ready figure
# Set journal-compliant defaults
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 7,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'lines.linewidth': 0.6,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.4,
    'ytick.major.width': 0.4,
})

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8))

# Panel A: Time series
annual = rainfall_series.resample('YE').sum()
axes[0].plot(annual.index.year, annual.values, color='#0072B2', lw=0.6, marker='o', ms=2)
axes[0].axhline(annual.mean(), color='red', ls='--', lw=0.5, label=f'Mean: {annual.mean():.0f} mm/yr')
axes[0].set_title('A: Annual Rainfall (Ethiopia)', loc='left')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('mm/year')
axes[0].legend(frameon=True, framealpha=0.8)
axes[0].grid(True, alpha=0.2)

# Panel B: Monthly climatology
monthly_clim = rainfall_series.groupby(rainfall_series.index.month).mean()
monthly_std = rainfall_series.groupby(rainfall_series.index.month).std()
axes[1].bar(monthly_clim.index, monthly_clim.values, color='#009E73', width=0.7,
            yerr=monthly_std.values, capsize=2, error_kw={'lw': 0.5})
axes[1].set_title('B: Monthly Climatology', loc='left')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('mm/month')
axes[1].set_xticks(range(1, 13))
axes[1].grid(True, axis='y', alpha=0.2)

plt.tight_layout()
fig.savefig('exports/journal_ready_figure.pdf', dpi=300, bbox_inches='tight')
fig.savefig('exports/journal_ready_figure.png', dpi=300, bbox_inches='tight')
plt.show()
print('Journal-ready figure exported to exports/')

## 8. Exercises

1. **Journal Match**: Using your target journal's author guidelines, create a figure with the exact column width specified. Export it as PDF with embedded fonts. 

2. **Accessibility Audit**: Take a figure you created earlier in this course. Re-plot it using a colorblind-friendly palette (viridis or Wong palette). Add markers/hatching so it's interpretable in grayscale. Show both versions side by side.

3. **CMYK Export**: For a figure of your choice, export as both RGB PNG and CMYK TIFF (use `savefig(..., format='tiff')` with appropriate metadata). Compare file sizes.

4. **Checklist Application**: Create a rainfall map of Ethiopia with all checklist items satisfied. Verify each item programmatically where possible.

### Mini-Project

**Submission-Ready Publication Figure**: Using CHIRPS data, create a complete 3-panel figure suitable for submission to a scientific journal:
- **Panel A**: Spatial map of mean annual rainfall over Ethiopia (use cartopy)
- **Panel B**: Time series of annual rainfall anomalies with ENSO years highlighted
- **Panel C**: Seasonal rainfall cycle for 3 contrasting years (wet, dry, average)

Requirements:
- Figure width = single-column (89 mm)
- 300 DPI PNG + PDF export
- Colorblind-friendly palette
- Clear, legible labels at print size
- All fonts embedded
- Create an `exports/` directory and save both formats